In [0]:
import os
import json
import random
import uuid
from datetime import datetime, timedelta
import pandas as pd

# 1. Match parameterized Bronze landing path
dbutils.widgets.text("landing_path", "/Volumes/workspace/bronze/raw_landing/qualtrics/responses/")
landing_path = dbutils.widgets.get("landing_path").rstrip("/") + "/"

# 2. Recursively search landing_path (and parent qualtrics folder if needed) for data files
def find_data_files(root_dir):
    matches = []
    if os.path.exists(root_dir):
        for dirpath, _, filenames in os.walk(root_dir):
            for f in filenames:
                if (
                    f.lower().endswith((".json", ".jsonl", ".ndjson", ".csv"))
                    and not f.startswith(("_", "."))
                    and "metadata" not in f.lower()
                ):
                    matches.append(os.path.join(dirpath, f))
    return sorted(matches)

valid_files = find_data_files(landing_path)
if not valid_files:
    parent_dir = os.path.dirname(landing_path.rstrip("/"))
    valid_files = find_data_files(parent_dir)

wrapper_key = None
is_json = True
is_json_array = False
file_ext = ".json"
target_dir = landing_path

if valid_files:
    sample_file = valid_files[0]
    target_dir = os.path.dirname(sample_file) + "/"
    file_ext = os.path.splitext(sample_file)[1].lower()
    is_json = file_ext in (".json", ".jsonl", ".ndjson")
    print(f"Found source file: {sample_file}")

    if is_json:
        with open(sample_file, "r", encoding="utf-8") as f:
            raw_text = f.read().strip()
        if raw_text.startswith("["):
            is_json_array = True
            template_df = pd.DataFrame(json.loads(raw_text))
        elif raw_text.startswith("{") and "\n" not in raw_text.splitlines()[0] and raw_text.endswith("}"):
            # Check if it's a single wrapped JSON object like {"responses": [...]} vs 1-line JSONL
            parsed = json.loads(raw_text)
            list_keys = [k for k, v in parsed.items() if isinstance(v, list) and len(v) > 0 and isinstance(v[0], dict)]
            if list_keys:
                wrapper_key = list_keys[0]
                template_df = pd.DataFrame(parsed[wrapper_key])
            else:
                template_df = pd.read_json(sample_file, lines=True)
        else:
            template_df = pd.read_json(sample_file, lines=True)
    else:
        template_df = pd.read_csv(sample_file)
else:
    # Fallback: read schema directly from your existing Bronze table if landing folder is empty
    bronze_tables = [r.tableName for r in spark.sql("SHOW TABLES IN workspace.bronze").collect()]
    assert len(bronze_tables) > 0, f"No files in {landing_path} and no tables found in workspace.bronze."
    bronze_tbl = f"workspace.bronze.{bronze_tables[0]}"
    print(f"Landing folder empty; reading schema from {bronze_tbl}")
    audit_cols = {"_rescued_data", "ingestion_timestamp", "ingested_at", "source_file", "batch_id", "_metadata"}
    spark_df = spark.table(bronze_tbl)
    keep_cols = [c for c in spark_df.columns if c.lower() not in audit_cols]
    template_df = spark_df.select(*keep_cols).limit(50).toPandas()

cols = list(template_df.columns)
print(f"Target directory: {target_dir} | Format: {file_ext} | Wrapper key: {wrapper_key}")
print(f"Detected columns: {cols}")

# 3. Reuse existing survey_ids so dim_survey & dataset_metadata.json stay 100% valid
survey_col = next((c for c in cols if "survey_id" in c.lower()), None)
existing_surveys = list(template_df[survey_col].dropna().unique()) if survey_col else ["SV_001"]

date_col = next((c for c in cols if "recorded" in c.lower() or "date" in c.lower() or "timestamp" in c.lower()), None)
sample_date_str = str(template_df[date_col].dropna().iloc[0]) if date_col and len(template_df[date_col].dropna()) > 0 else ""
use_iso_t = "T" in sample_date_str

departments = ["Finance", "Operations", "People", "Risk Consulting", "Technology", "Legal & Compliance"]
regions = ["Asia Pacific", "Europe", "Latin America", "North America"]
resp_types = ["Client", "Employee", "Partner"]

# 4. Generate 35 new realistic responses (including null ratings & new department)
rows = []
base_time = datetime.now()

for _ in range(35):
    completed = random.choices([1, 0], weights=[0.8, 0.2])[0]
    rating = random.randint(1, 10) if completed == 1 else None
    dt_obj = base_time - timedelta(days=random.randint(0, 14), hours=random.randint(0, 23))
    rec_dt = dt_obj.strftime("%Y-%m-%dT%H:%M:%SZ") if use_iso_t else dt_obj.strftime("%Y-%m-%d %H:%M:%S")

    base_row = template_df.iloc[random.randint(0, len(template_df) - 1)].to_dict() if len(template_df) > 0 else {}

    for c in cols:
        cl = c.lower()
        if "response_id" in cl or cl == "responseid":
            base_row[c] = f"R_SYNTH_{uuid.uuid4().hex[:10].upper()}"
        elif "survey_id" in cl:
            base_row[c] = random.choice(existing_surveys)
        elif "department" in cl:
            base_row[c] = random.choice(departments)
        elif "region" in cl:
            base_row[c] = random.choice(regions)
        elif "respondent_type" in cl:
            base_row[c] = random.choice(resp_types)
        elif "rating" in cl or "score" in cl:
            base_row[c] = rating
        elif "completed" in cl or "finished" in cl:
            base_row[c] = bool(completed) if isinstance(base_row.get(c), bool) else completed
        elif "recorded" in cl or "end_date" in cl or "timestamp" in cl:
            base_row[c] = rec_dt

    rows.append(base_row)

batch_df = pd.DataFrame(rows)

# 5. Append 5 intentional duplicate rows to stress-test Silver deduplication (40 total)
messy_batch_df = pd.concat([batch_df, batch_df.head(5)], ignore_index=True)

os.makedirs(target_dir, exist_ok=True)
ts_tag = datetime.now().strftime("%Y%m%d_%H%M%S")
out_file = os.path.join(target_dir, f"qualtrics_synth_batch_{ts_tag}{file_ext}")

if is_json:
    records = json.loads(messy_batch_df.to_json(orient="records"))
    if wrapper_key:
        with open(out_file, "w", encoding="utf-8") as f:
            json.dump({wrapper_key: records}, f, indent=2)
    elif is_json_array:
        with open(out_file, "w", encoding="utf-8") as f:
            json.dump(records, f, indent=2)
    else:
        messy_batch_df.to_json(out_file, orient="records", lines=True)
else:
    messy_batch_df.to_csv(out_file, index=False)

print(f"Success! Wrote {len(messy_batch_df)} rows (35 unique + 5 duplicates) to:\n{out_file}")